# 11: Baselines (RQ3)

Design decisions
- Frozen configurations, box tables and candidate sets of notebooks 05 and 08, hashes asserted. Targets as frozen: alpha_0 = 0.05 for negative-outcome claims, alpha_1 = 0.70 for enrichment claims, delta_1 = delta_2 = 0.025, alpha_sel per task and kind.
- Real data, 20 re-draws (seeds 11000 to 11019): the explained patients of calibration split 2 stay in split 2, and a new split 1 of the original size is drawn at random from the other records of splits 1 and 2. Each re-draw is a random draw of the original design, and the policy test uses the same explained patients as notebook 09. Every method uses the same labelled records: split 1 of the re-draw and the explained patients of split 2. Results are on the 5,000 explained test patients; each released box's claim is also checked on the full test split, which no method sees: the share of released patients whose box has test error above the target, and the share whose box is refuted (exact lower 95% limit, Bonferroni-corrected over the released boxes, above the target).
- Methods on real data: SCAX; per-box bound without prior (B4); empirical error rule (B5); uncertified order-0 AXp and widest inflated box (B1, B2), for all patients and at SCAX's coverage (most confident first); RuleFit rule dictionary with the same bound and a uniform prior over the dictionary (B6), and the same restricted to rules that are sound for the model; SGR and Learn-then-Test per kind at alpha_sel and delta_2 / 2 (B9, B10); split and Mondrian conformal prediction at epsilon = 0.05, releasing singleton sets (B11).
- Paired Wilcoxon signed-rank tests over the 20 re-draws, Holm correction across baselines, median paired differences.
- Planted-rule population (true risks known): rule P4, whose risk varies smoothly, with the model and candidate cache of notebook 10. Rules P1 to P3 are not used here: every box that follows such a rule has true risk equal to the noise level, so every method's claims are true there. Two target pairs: alpha_0 = 0.05 with alpha_1 = 0.70 (as on real data) and alpha_0 = 0.02 with alpha_1 = 0.50. 200 calibration draws of 10,000 with replacement, M = 10, mixture prior. Coverage against true claim precision for SCAX, B4, B5, B1, B2, B6 and the confidence-aware AXp of Chagas et al. (2026) with confidence floors 1 - alpha_y (B8).
- SHAP, LIME and Anchors (B12, descriptive) on 200 explained test patients per task: the smallest number of top-ranked SHAP or LIME features that forms a sufficient box, and whether each anchor's smallest containing grid box is sound. SHAP uses the native model retrained with the frozen settings, only if its export hash equals the frozen model hash.
- Intermediate results are saved on Drive after every re-draw and setting, so an interrupted run resumes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver', 'imodels', 'lime', 'anchor-exp'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, math, pickle, time, warnings
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits, planted
from scax.experiments.main_results import FrozenTask
from scax.experiments.baselines import RealComparison, SyntheticComparison, paired_tests
from scax.experiments.validity import build_setting
from scax.explain.candidates import CandidateGenerator, SoundnessOracle
from scax.baselines import rule_dictionary as rd
from scax.baselines.confidence_axp import ConfidenceAXp
warnings.filterwarnings('ignore', category=FutureWarning)
quiet_warnings()

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
REC08 = json.load(open(f'{paths.RESULTS}/08_certificates.json'))
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'
BASE_DIR = paths.DERIVED / 'baselines'; BASE_DIR.mkdir(parents=True, exist_ok=True)
REDRAWS, REDRAW_SEED0, N_RULE_FIT, MAX_RULES = 20, 11000, 20000, 200
SYN_TARGETS, R_SYN, N1_SYN, M_SYN, N_ATTR = [{0: 0.05, 1: 0.70}, {0: 0.02, 1: 0.50}], 200, 10000, 10, 200


def load_task(task):
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == FT[task].cfg['split_sha256'], 'split differs from the frozen one'
    return X.to_numpy(dtype=float), y, code


def generator(ft):
    c = ft.cfg['candidates']
    gen = CandidateGenerator(ft.pred, M=c['M'], seed=c['order_seed'], node_budget=c['node_budget'])
    assert gen.load(CERT_DIR / f'{ft.task}__candidates.pkl')
    return gen


for t in TASKS:
    assert sha256_file(CERT_DIR / f'{t}__boxes.parquet') == REC08['box_table_sha256'][t], 'box table differs from notebook 08'
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| alpha', FT[t].targets['alpha_0'], FT[t].targets['alpha_1'],
          '| alpha_sel', FT[t].targets['alpha_sel'])

## Real data: 20 calibration re-draws

In [ ]:
REAL, RULES = [], {}
for task in TASKS:
    ft = FT[task]
    Xa, y, code = load_task(task)
    t0 = time.time()
    rc = RealComparison(ft, pd.read_parquet(CERT_DIR / f'{task}__boxes.parquet'), generator(ft), Xa, y, code,
                        ft.targets['delta_1'], ft.targets['delta_2'])
    status_file = BASE_DIR / f'{task}__second_check_status.npy'
    if status_file.exists():
        st = np.load(status_file)
        assert len(st) == len(rc.status) and np.all(st[rc.status != 0] == rc.status[rc.status != 0])
        rc.status = st
    rules_file = BASE_DIR / f'{task}__rules.pkl'
    if rules_file.exists():
        rules = pickle.load(open(rules_file, 'rb'))
    else:
        tr = splits.indices(code, 'train')
        sub = np.sort(np.random.default_rng(SEED).choice(tr, N_RULE_FIT, replace=False))
        rules = rd.fit_rulefit(Xa[sub], y[sub], ft.ens.feature_names, max_rules=MAX_RULES, seed=SEED)
        quiet_warnings()
        pickle.dump(rules, open(rules_file, 'wb'))
    rc.set_rules(rules, SoundnessOracle(ft.pred))
    RULES[task] = rules
    print(f'{task}: set up in {time.time() - t0:.0f}s; {len(rules)} RuleFit rules (mean {np.mean([r.size for r in rules]):.1f} conditions), '
          f'{int(rc.r_sound.sum())} of {len(rc.r_sound)} rule claims sound for the model')
    out_file = BASE_DIR / f'{task}__redraws_v2.parquet'
    done = pd.read_parquet(out_file) if out_file.exists() else pd.DataFrame()
    for r in range(REDRAWS):
        if len(done) and r in set(done.redraw):
            continue
        t1 = time.time()
        res, n_ver = rc.run(r, REDRAW_SEED0 + r)
        res.insert(0, 'task', task)
        done = pd.concat([done, res], ignore_index=True)
        done.to_parquet(out_file, index=False)
        np.save(status_file, rc.status)
        s = res[res.method == 'SCAX']
        print(f'  re-draw {r}: SCAX coverage {s.coverage.iloc[0]:.3f} / {s.coverage.iloc[1]:.3f}, '
              f'policy {bool(s.policy_certified.iloc[0])} / {bool(s.policy_certified.iloc[1])}, newly checked boxes {n_ver}, {time.time() - t1:.0f}s')
    REAL.append(done)
    del rc
REAL = pd.concat(REAL, ignore_index=True)
REAL.to_csv(f'{paths.RESULTS}/11_baselines.csv', index=False)

In [ ]:
KIND = {0: 'negative outcome', 1: 'enrichment'}
agg = {'coverage': ('coverage', 'mean'), 'coverage_lo': ('coverage', lambda s: s.quantile(0.025)),
       'coverage_hi': ('coverage', lambda s: s.quantile(0.975)), 'error_rate': ('error_rate', 'mean'),
       'mean_size': ('mean_size', 'mean'), 'median_support': ('median_support', 'mean'),
       'above_target_share': ('above_target_share', 'mean'), 'refuted_share': ('refuted_share', 'mean'),
       'policy_certified': ('policy_certified', 'mean'), 'sound_share': ('sound_share', 'mean')}
REAL['policy_certified'] = REAL['policy_certified'].astype(float)
SUMMARY = REAL.groupby(['task', 'kind', 'method'], sort=False).agg(**agg).reset_index()
SUMMARY['kind'] = SUMMARY['kind'].map(KIND)
SUMMARY.to_csv(f'{paths.RESULTS}/11_baselines_summary.csv', index=False)
for (task, kind), g in SUMMARY.groupby(['task', 'kind'], sort=False):
    print(f'\n{NAMES[task]}, {kind} (means over {REDRAWS} re-draws; policy_certified = share of re-draws)')
    print(g.drop(columns=['task', 'kind']).round(4).to_string(index=False))

In [ ]:
TESTS = pd.concat([paired_tests(REAL, m) for m in ('coverage', 'above_target_share', 'refuted_share', 'error_rate')], ignore_index=True)
TESTS['kind'] = TESTS['kind'].map(KIND)
TESTS.to_csv(f'{paths.RESULTS}/11_paired_tests.csv', index=False)
view = TESTS[TESTS.metric.isin(['coverage', 'above_target_share'])]
print(view[['task', 'kind', 'metric', 'baseline', 'reference_mean', 'baseline_mean', 'median_difference', 'p_holm']].round(4).to_string(index=False))

## Planted-rule population: coverage against true claim precision

In [ ]:
XP = pd.read_parquet(paths.DERIVED / 'planted_population.parquet')
P4_B0 = planted.p4_intercept(XP)
MODEL10 = {'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 50, 'gamma': 2, 'max_bin': 32}
SETTINGS10 = [(r, e) for r in ('P1', 'P2', 'P3') for e in planted.NOISE_LEVELS] + [('P4', 0.0)]
SYN, SYN_DIAG, st = [], [], None
for alpha in SYN_TARGETS:
    tag = f'P4_alpha{alpha[0]:g}_{alpha[1]:g}'
    f = BASE_DIR / f'synthetic_{tag}.parquet'
    if f.exists():
        SYN.append(pd.read_parquet(f)); print(tag, 'loaded from Drive'); continue
    t0 = time.time()
    if st is None:
        s_i = SETTINGS10.index(('P4', 0.0))
        args = (XP, 'P4', 0.0, P4_B0, MODEL10, 40000, 20, SEED + s_i, {0: 0.05, 1: 0.70}, 0.05)
        try:
            st, diag = build_setting(*args, lam=0.5, cache_path=paths.DERIVED / 'cache' / '10_P4_eps0.00.pkl', z3_n=0)
        except ValueError as e:                     # notebook 10's cache belongs to a different classifier build
            print('notebook 10 cache not usable:', e, '- building candidate sets again')
            st, diag = build_setting(*args, lam=0.5, cache_path=BASE_DIR / 'P4_candidates.pkl', z3_n=0)
        X = XP.to_numpy(dtype=float)
        sub = np.random.default_rng(SEED).choice(len(st.idx_train), N_RULE_FIT, replace=False)
        syn_rules = rd.fit_rulefit(X[st.idx_train[sub]], st.y_train[sub], list(XP.columns), max_rules=MAX_RULES, seed=SEED)
        quiet_warnings()
        oracle = SoundnessOracle(st.pred)
        syn_sound = [oracle.sound(rd.outer_box(r, st.grid), c) for r in syn_rules for c in (0, 1)]
    sc = SyntheticComparison(st, M_SYN, 0.025, alpha=alpha)
    sc.add_confidence_axp(ConfidenceAXp(st.pred.ens, {0: 1 - alpha[0], 1: 1 - alpha[1]}))
    sc.add_rules(syn_rules, syn_sound)
    df = sc.run(R_SYN, N1_SYN, seed=2000 + int(1000 * alpha[0]))
    df.insert(0, 'alpha_1', alpha[1]); df.insert(0, 'alpha_0', alpha[0])
    df.to_parquet(f, index=False)
    SYN.append(df)
    SYN_DIAG.append({'targets': [alpha[0], alpha[1]], 'cells': diag['cells'], 'boxes': diag['boxes'], 'rules': len(syn_rules),
                     'sound_rule_claims': int(sum(syn_sound)), 'seconds': time.time() - t0})
    print(f"{tag}: {diag['cells']} cells, {diag['boxes']} boxes, {len(syn_rules)} rules "
          f"({int(sum(syn_sound))} of {len(syn_sound)} rule claims sound for the model), {time.time() - t0:.0f}s")
SYN = pd.concat(SYN, ignore_index=True)
SYN.to_csv(f'{paths.RESULTS}/11_synthetic_claims.csv', index=False)
SYN_SUM = SYN.groupby(['alpha_0', 'alpha_1', 'method'], sort=False).agg(
    coverage=('coverage', 'mean'), coverage_lo=('coverage', lambda s: s.quantile(0.025)), coverage_hi=('coverage', lambda s: s.quantile(0.975)),
    claim_precision=('claim_precision', 'mean'), precision_lo=('claim_precision', lambda s: s.quantile(0.025)),
    precision_hi=('claim_precision', lambda s: s.quantile(0.975)), any_false_claim=('claim_violation', 'mean'),
    true_selective_risk=('true_selective_risk', 'mean'), mean_size=('mean_size', 'mean')).reset_index()
SYN_SUM.to_csv(f'{paths.RESULTS}/11_synthetic_summary.csv', index=False)
print(SYN_SUM.round(4).to_string(index=False))

## SHAP, LIME and Anchors: do they form sufficient boxes? (descriptive)

In [ ]:
import xgboost as xgb
from lime.lime_tabular import LimeTabularExplainer
from anchor import anchor_tabular
from scax.models import train
from scax.models.export import export_model
from scax.baselines.attribution import anchor_rule, smallest_sufficient_k

ATTR = []
for task in TASKS:
    ft = FT[task]; names = list(ft.ens.feature_names)
    Xa, y, code = load_task(task)
    tr = splits.indices(code, 'train')
    native = train.fit('xgboost', Xa[tr], y[tr], ft.cfg['model']['params'], SEED)
    ens = export_model(native, 'xgboost', names); ens.tau = ft.ens.tau
    shap_ok = ens.hash() == ft.cfg['model']['sha256']
    rng = np.random.default_rng(SEED + 12)
    rows = ft.test_rows[np.sort(rng.choice(len(ft.test_rows), N_ATTR, replace=False))]
    Xp = Xa[rows]; C = ft.grid.cells(Xp); yh = ft.pred.predict_cells(C).astype(int)
    oracle = SoundnessOracle(ft.pred)
    gen = generator(ft)
    contrib = native.get_booster().predict(xgb.DMatrix(Xp), pred_contribs=True)[:, :-1] if shap_ok else None
    bg = Xa[np.sort(rng.choice(tr, 5000, replace=False))]
    prob = lambda Z: (lambda p: np.c_[1 - p, p])(1 / (1 + np.exp(-ft.pred.score(np.asarray(Z, dtype=float)))))
    lime_ex = LimeTabularExplainer(bg, feature_names=names, class_names=['no', 'yes'], discretize_continuous=True, random_state=SEED)
    anc_ex = anchor_tabular.AnchorTabularExplainer(['no', 'yes'], names, bg)
    try:
        bins = {f: lam.__defaults__[0] for f, lam in anc_ex.disc.lambdas.items()}
    except Exception:
        bins = None
    np.random.seed(SEED)
    t0 = time.time()
    for i in range(N_ATTR):
        row = {'task': task, 'row': int(rows[i]), 'predicted': int(yh[i]),
               'smallest_candidate_size': min(c.size for c in gen.cache[gen.cell_key(C[i])])}
        if shap_ok:
            rank = list(np.argsort(-np.abs(contrib[i]), kind='stable'))
            row['shap_k'] = smallest_sufficient_k(oracle, ft.grid, C[i], int(yh[i]), rank)
        e = lime_ex.explain_instance(Xp[i], prob, num_features=len(names), num_samples=2000)
        row['lime_k'] = smallest_sufficient_k(oracle, ft.grid, C[i], int(yh[i]), [j for j, _ in e.as_map()[1]])
        try:
            a = anc_ex.explain_instance(Xp[i], lambda Z: ft.pred.predict(np.asarray(Z, dtype=float)), threshold=0.95)
            r = anchor_rule(a.names(), names, bins)
            row.update({'anchor_size': len(a.names()), 'anchor_precision': float(a.precision()),
                        'anchor_sound': bool(oracle.sound(rd.outer_box(r, ft.grid), int(yh[i])))})
        except Exception as ex:
            row.update({'anchor_size': None, 'anchor_precision': None, 'anchor_sound': None, 'anchor_error': str(ex)[:80]})
        ATTR.append(row)
    print(f'{task}: SHAP from the native model {"used (hash equal)" if shap_ok else "not used (hash differs)"}; '
          f'{N_ATTR} patients in {time.time() - t0:.0f}s')
ATTR = pd.DataFrame(ATTR)
ATTR.to_csv(f'{paths.RESULTS}/11_attribution_sufficiency.csv', index=False)
ATTR_SUM = []
for task, g in ATTR.groupby('task', sort=False):
    s = {'task': task, 'patients': len(g), 'median_smallest_candidate_size': float(g.smallest_candidate_size.median())}
    for m in ('shap_k', 'lime_k'):
        if m in g and g[m].notna().any():
            k = g[m].astype(float)
            s[f'{m}_median'] = float(k.median())
            s[f'{m[:-2]}_top5_sufficient'] = float((k <= 5).mean())
    a = g[g.anchor_sound.notna()]
    s.update({'anchors_run': len(a), 'anchor_median_size': float(a.anchor_size.median()) if len(a) else None,
              'anchor_mean_precision': float(a.anchor_precision.mean()) if len(a) else None,
              'anchor_sound_share': float(a.anchor_sound.astype(bool).mean()) if len(a) else None})
    ATTR_SUM.append(s)
ATTR_SUM = pd.DataFrame(ATTR_SUM)
print(ATTR_SUM.round(4).T.to_string())

## Figures

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
STYLE = {'SCAX': ('#0072B2', 'o'), 'Per-box bound without prior': ('#D55E00', 's'), 'Empirical error rule': ('#CC79A7', 'D'),
         'Uncertified AXp': ('#E69F00', '^'), 'Uncertified inflated box': ('#56B4E9', 'v'),
         'Rule dictionary': ('#009E73', 'P'), 'Rule dictionary, sound rules only': ('#009E73', 'X'),
         'Confidence-aware AXp': ('#000000', '*')}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.3), sharey=True)
for ax, alpha in zip(axes, SYN_TARGETS):
    s = SYN_SUM[(SYN_SUM.alpha_0 == alpha[0]) & (SYN_SUM.alpha_1 == alpha[1])]
    for _, r in s.iterrows():
        col, mk = STYLE[r.method]
        ax.errorbar(r.coverage, r.claim_precision, xerr=[[r.coverage - r.coverage_lo], [r.coverage_hi - r.coverage]],
                    yerr=[[r.claim_precision - r.precision_lo], [r.precision_hi - r.claim_precision]],
                    fmt=mk, color=col, ms=7 if r.method == 'SCAX' else 5, mfc=col if 'sound' not in r.method else 'white',
                    lw=1, capsize=0, label=r.method)
    ax.set_title(f'Targets: {alpha[0]:.0%} for negative-outcome claims,\n{alpha[1]:.0%} for enrichment claims', fontsize=8.5)
    ax.set_xlabel('Share of the population given an explanation'); ax.set_xlim(-0.02, 1.02)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_ylabel('Share of released explanations\nwhose claim is true')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.01), fontsize=7.5)
fig.tight_layout(rect=(0, 0.16, 1, 1))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_coverage_vs_true_claim_precision.{ext}', dpi=600, bbox_inches='tight')
plt.show()

SHOW = ['SCAX', 'Per-box bound without prior', 'Empirical error rule', 'Uncertified AXp, all patients',
        'Uncertified AXp, matched coverage', 'Uncertified inflated box, all patients', 'Rule dictionary']
LABEL = {'Uncertified AXp, all patients': 'Uncertified AXp (all)', 'Uncertified AXp, matched coverage': 'Uncertified AXp (SCAX coverage)',
         'Uncertified inflated box, all patients': 'Uncertified inflated box (all)'}
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.0), sharex=True)
for i, task in enumerate(TASKS):
    for j, kind in enumerate(['negative outcome', 'enrichment']):
        ax = axes[i, j]
        s = SUMMARY[(SUMMARY.task == task) & (SUMMARY.kind == kind)].set_index('method').reindex(SHOW)
        yv = np.arange(len(SHOW))[::-1]
        ax.barh(yv, s.above_target_share.fillna(0), color=['#0072B2' if m == 'SCAX' else '#999999' for m in SHOW], height=0.6)
        ax.set_yticks(yv); ax.set_yticklabels([LABEL.get(m, m) for m in SHOW] if j == 0 else [])
        ax.set_title(f'{NAMES[task]}, {kind} claims', fontsize=8.5)
        for m, y0, v, c, snd in zip(SHOW, yv, s.above_target_share.fillna(0), s.coverage, s.sound_share):
            extra = f'; sound for the model: {snd:.0%}' if m == 'Rule dictionary' and snd == snd else ''
            ax.text(v + 0.01, y0, f'{v:.0%}   (explained: {c:.0%}{extra})', va='center', fontsize=6.5)
        ax.set_xlim(0, 1); ax.set_xticks([0, 0.25, 0.5, 0.75]); ax.xaxis.set_major_formatter(PercentFormatter(1.0))
fig.supxlabel('Share of released explanations whose box has test error above its target', fontsize=8.5)
fig.tight_layout()
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_baselines_claims_real.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
key = SUMMARY[SUMMARY.method.isin(['SCAX', 'Rule dictionary', 'Rule dictionary, sound rules only', 'SGR', 'Uncertified AXp, all patients'])]
write_json(f'{paths.RESULTS}/11_baselines.json', {
    'seed': SEED, 'redraws': REDRAWS, 'redraw_seeds': [REDRAW_SEED0, REDRAW_SEED0 + REDRAWS - 1],
    'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'box_table_sha256': REC08['box_table_sha256'],
    'rules': {t: len(RULES[t]) for t in TASKS}, 'synthetic': {'rule': 'P4', 'targets': [[a[0], a[1]] for a in SYN_TARGETS], 'repetitions': R_SYN,
                                                              'n1': N1_SYN, 'M': M_SYN, 'diagnostics': SYN_DIAG},
    'key_rows': key.to_dict('records'), 'attribution': ATTR_SUM.to_dict('records')})
print('written: results/11_baselines.csv, 11_baselines_summary.csv, 11_paired_tests.csv, 11_synthetic_claims.csv, '
      '11_synthetic_summary.csv, 11_attribution_sufficiency.csv, 11_baselines.json, figures')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "11: baselines rerun with the corrected re-draw design (explained patients stay in calibration split 2), paired tests, synthetic claim precision, attribution sufficiency, figures"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)